# Parkinson’s Disease Classification
## UCI Machine Learning Repository
This notebook loads **multiple UCI Parkinson’s datasets**, aligns shared features (Strategy B), merges them, and trains a PyTorch model.

In [1]:

# Install dependencies (run once)
!pip install ucimlrepo


In [2]:
# Imports
import os

PROJECT_ROOT = os.getcwd()
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [3]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# ===============================
# UCI DATASET LOADER (NO HEADERS)
# ===============================
def fetch_uci_dataset(dataset_id, drop_cols=None):
    dataset = fetch_ucirepo(id=dataset_id)

    X = dataset.data.features.copy()
    y = dataset.data.targets.copy()

    if drop_cols:
        X = X.drop(columns=drop_cols, errors="ignore")

    # Ensure y is 1D
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]

    # Convert to NumPy — NO column renaming
    X = X.to_numpy()
    y = y.to_numpy()

    return X, y, dataset.metadata


# ===============================
# TXT FILE LOADER (NO HEADERS)
# ===============================
def load_txt_features(txt_path):
    df = pd.read_csv(txt_path, header=None)

    # Last column = label
    y = df.iloc[:, -1]
    X_raw = df.iloc[:, :-1]

    # Map known columns into unified schema
    col_map = {
        0:  'MDVP:Jitter(%)',
        1:  'MDVP:Jitter(Abs)',
        2:  'MDVP:RAP',
        3:  'MDVP:PPQ',
        4:  'Jitter:DDP',
        5:  'MDVP:Shimmer',
        6:  'MDVP:Shimmer(dB)',
        7:  'Shimmer:APQ3',
        8:  'Shimmer:APQ5',
        10: 'Shimmer:DDA',
        12: 'NHR',
        13: 'HNR',
        15: 'MDVP:Fo(Hz)',
        17: 'MDVP:Flo(Hz)',
        18: 'MDVP:Fhi(Hz)'
    }

    existing = [c for c in col_map if c in X_raw.columns]
    X = X_raw[existing].rename(columns=col_map)

    # Align to unified schema
    X = align_to_unified(X)

    return X, y



# ===============================
# LOAD DATASETS
# ===============================
# UCI Parkinsons (main)
X1, y1, _ = fetch_uci_dataset(174, drop_cols=["name"])

# UCI Parkinsons Telemonitoring
X2, y2_raw, _ = fetch_uci_dataset(189, drop_cols=["subject#"])
y2 = (y2_raw > 0).astype(int)  # binarize

# TXT dataset
X3, y3 = load_txt_features("/Users/sanjithsridharan/PycharmFiles/train_data.txt")

# ===============================
# SANITY CHECK
# ===============================
print("UCI 174:", X1.shape, y1.shape)
print("UCI 189:", X2.shape, y2.shape)
print("TXT:", X3.shape, y3.shape)

ValueError: TXT file must contain exactly 22 features, found 28

In [ ]:
# Concatenate all X dataframes. They should now have consistent columns and order.
X = pd.concat([X1, X2, X3], ignore_index=True)
y = pd.concat([y1, y2, y3], ignore_index=True)

# Important: Handle potential NaNs introduced by feature alignment or missing data.
# For now, a simple mean imputation. More sophisticated methods can be used.
# This is crucial before scaling, as StandardScaler cannot handle NaNs.
X.fillna(X.mean(numeric_only=True), inplace=True)

print("Merged dataset shape:", X.shape)
print("Merged X head:\n", X.head())
print("Merged y head:\n", y.head())

In [ ]:
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)


In [ ]:
import torch
import torch.nn as nn


class ParkinsonNet(nn.Module):
    def __init__(self, input_dim=22):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


def load_model(model_path: str):
    model = ParkinsonNet()
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()
    return model


In [ ]:
best_val_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t).squeeze()
    loss = criterion(outputs, y_train_t.float())
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_t).squeeze()
        val_loss = criterion(val_outputs, y_test_t.float())

    print(f"Epoch {epoch+1} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = model.state_dict()
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
import matplotlib.pyplot as plt

train_losses = []
val_losses = []

best_val_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(200):
    # ---- Training ----
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t).squeeze()
    loss = criterion(outputs, y_train_t.float())
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    # ---- Validation ----
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_t).squeeze()
        val_loss = criterion(val_outputs, y_test_t.float())

    val_losses.append(val_loss.item())

    print(f"Epoch {epoch+1} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

    # ---- Early stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = model.state_dict()



In [ ]:

# Evaluation
with torch.no_grad():
    preds = (model(X_test_t).squeeze() > 0.5).int().numpy()

# Binarize y_test to match classification predictions
y_test_binarized = (y_test > 0).astype(int)

print("Accuracy:", accuracy_score(y_test_binarized, preds))
print(classification_report(y_test_binarized, preds))


In [ ]:
# Save trained model and scaler
torch.save(model.state_dict(), "parkinsons_model.pth")

import joblib
joblib.dump(scaler, "scaler.pkl")

# Save feature order (CRITICAL)
feature_columns = list(X.columns)
joblib.dump(feature_columns, "features.pkl")


In [ ]:
import librosa
import numpy as np


def audio_to_features(audio_file) -> np.ndarray:
    y, sr = librosa.load(audio_file, sr=None)

    # Fundamental frequency
    f0 = librosa.yin(y, fmin=50, fmax=500)
    f0 = f0[np.isfinite(f0)]

    jitter = np.std(f0) / np.mean(f0)
    shimmer = np.std(np.abs(y))

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_means = mfccs.mean(axis=1)

    features = np.array([
        np.mean(f0), np.max(f0), np.min(f0),
        jitter, jitter,
        jitter * 0.8, jitter * 0.7, jitter * 1.2,
        shimmer, shimmer,
        shimmer * 0.7, shimmer * 0.9,
        shimmer * 1.1, shimmer * 1.3,
        0.02,  # NHR proxy
        20.0,  # HNR proxy
        mfcc_means[0], mfcc_means[1],
        mfcc_means[2], mfcc_means[3],
        mfcc_means[4], mfcc_means[5]
    ])

    return features.reshape(1, -1)


In [ ]:
def predict_from_audio(audio_path, model, scaler, feature_columns):
    features_df = audio_to_features(audio_path, feature_columns)
    features_scaled = scaler.transform(features_df)

    with torch.no_grad():
        tensor = torch.tensor(features_scaled, dtype=torch.float32)
        prob = model(tensor).item()

    return prob, features_df

In [ ]:
# Install pydub and ffmpeg (pydub backend) if not already installed
!pip install pydub
!apt-get install -y ffmpeg

from google.colab import files
from pydub import AudioSegment

uploaded = files.upload()
original_audio_path = list(uploaded.keys())[0]

# Always convert the uploaded audio to a standard WAV format
try:
    audio = AudioSegment.from_file(original_audio_path)
    wav_audio_path = original_audio_path.rsplit('.', 1)[0] + '_converted.wav'
    audio.export(wav_audio_path, format="wav")
    audio_path = wav_audio_path
except Exception as e:
    print(f"Error converting audio file: {e}")
    audio_path = original_audio_path # Fallback to original if conversion fails

# Correctly call predict_from_audio with all required arguments
prob, features_df = predict_from_audio(audio_path, model, scaler, feature_columns)

# Ensure probability is within a sensible range for display
prob = max(min(prob, 0.001), 0.999)

print(f"Parkinson's likelihood: {prob*100:.2f}%")
print(features_df)

In [ ]:
!pip install streamlit cloudflared

import streamlit as st

In [ ]:
%%writefile app.py
import streamlit as st
import tempfile
import torch
import joblib
# from model import ParkinsonNet # Removed this line
from sklearn.preprocessing import StandardScaler # Ensure StandardScaler is imported here as well

# Neural network (moved here from cell 3f959ffd for app.py)
class ParkinsonNet(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# Load model, scaler, features
# -----------------------------
# X_train is not available in app.py context, so load scaler directly if it exists, otherwise it will fail here.
# The scaler should be saved in `scaler.pkl` from the notebook's execution.
# If not found, `FileNotFoundError` will be caught and the app will indicate a warning.

# The notebook state shows that X_train, model, scaler, and feature_columns were already defined and saved correctly.
# We need to make sure `scaler` is loaded from the saved file, not re-initialized.

try:
    scaler = joblib.load("scaler.pkl")
except FileNotFoundError:
    st.error("scaler.pkl not found. Please ensure the training and saving steps are executed correctly.")
    st.stop() # Stop the app if essential components are missing

try:
    feature_columns = joblib.load("features.pkl")
except FileNotFoundError:
    st.error("features.pkl not found. Please ensure the training and saving steps are executed correctly.")
    st.stop()

model = ParkinsonNet(input_dim=len(feature_columns))
model.load_state_dict(torch.load("parkinsons_model.pth", map_location="cpu"))
model.eval()

# Define the predict_from_audio function directly in app.py for Streamlit
import librosa
import numpy as np
import pandas as pd
from scipy.stats import variation

def audio_to_features(audio_path, feature_columns, sr=22050):
    # Load audio (mono)
    y, sr = librosa.load(audio_path, sr=sr, mono=True)

    if len(y) < sr * 0.3:
        raise ValueError("Audio too short for reliable feature extraction.")

    features = {}

    # Pitch extraction (modern replacement for Praat pitch)
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=75,
        fmax=500,
        sr=sr
    )

    f0_voiced = f0[~np.isnan(f0)]

    if len(f0_voiced) == 0:
        raise ValueError("No voiced segments detected.")

    features["MDVP:Fo(Hz)"] = float(np.mean(f0_voiced))
    features["MDVP:Fhi(Hz)"] = float(np.max(f0_voiced))
    features["MDVP:Flo(Hz)"] = float(np.min(f0_voiced))

    # Jitter approximations (from F0 instability)
    features["MDVP:Jitter(%)"] = float(variation(f0_voiced) * 100)

    if len(f0_voiced) > 2:
        ddp = np.mean(np.abs(np.diff(f0_voiced, n=2)))
    else:
        ddp = 0.0
    features["Jitter:DDP"] = float(ddp)

    # Shimmer approximations (from amplitude envelope)
    rms = librosa.feature.rms(y=y)[0]

    if len(rms) > 1:
        shimmer_local = np.mean(np.abs(np.diff(rms))) / np.mean(rms)
    else:
        shimmer_local = 0.0

    features["MDVP:Shimmer"] = float(shimmer_local)
    features["Shimmer:APQ3"] = float(np.mean(np.abs(np.diff(rms, n=1))) if len(rms) > 3 else 0.0)
    features["Shimmer:APQ5"] = float(np.mean(np.abs(np.diff(rms, n=2))) if len(rms) > 5 else 0.0)
    features["Shimmer:DDA"] = float(3 * features["Shimmer:APQ3"])

    # Harmonics-to-Noise Ratio (HNR approximation)
    y_harmonic, y_percussive = librosa.effects.hpss(y)

    harm_energy = np.sum(y_harmonic ** 2)
    noise_energy = np.sum(y_percussive ** 2) + 1e-10

    features["HNR"] = float(10 * np.log10(harm_energy / noise_energy))

    # Noise-to-Harmonics Ratio (NHR)
    if "NHR" in feature_columns:
        if features["HNR"] > 0:
            features["NHR"] = float(1.0 / features["HNR"])
        else:
            features["NHR"] = 0.0

    # Final safety checks
    all_features = {col: features.get(col, 0.0) for col in feature_columns}
    values = np.array(list(all_features.values()), dtype=float)

    if np.isnan(values).any() or np.all(values == 0):
        raise ValueError("Invalid recording: sustained vowel required.")

    return pd.DataFrame([values], columns=feature_columns)

def predict_from_audio(audio_path, model, scaler, feature_columns):
    features_df = audio_to_features(audio_path, feature_columns)
    features_scaled = scaler.transform(features_df)

    with torch.no_grad():
        tensor = torch.tensor(features_scaled, dtype=torch.float32)
        prob = torch.sigmoid(model(tensor)).item() # Apply sigmoid to get probability

    return prob, features_df


# -----------------------------
# Streamlit App
# -----------------------------

import streamlit as st
import torch
import joblib
import numpy as np

from model import load_model
from audio_features import audio_to_features
from data_utils import load_txt_features

st.set_page_config(page_title="Parkinson's Detection", layout="centered")

st.title("🧠 Parkinson’s Disease Detection")
st.write("Upload **both** an audio recording and a TXT feature file.")

# Uploads
audio_file = st.file_uploader("Upload voice recording (.wav)", type=["wav"])
txt_file = st.file_uploader("Upload feature TXT file", type=["txt"])

if audio_file and txt_file:
    try:
        audio_features = audio_to_features(audio_file)
        txt_features = load_txt_features(txt_file)

        # Combine (mean fusion)
        X = np.mean([audio_features, txt_features], axis=0)

        scaler = joblib.load("scaler.pkl")
        X = scaler.transform(X)

        model = load_model("parkinson_model.pt")

        with torch.no_grad():
            logits = model(torch.tensor(X, dtype=torch.float32))
            prob = torch.sigmoid(logits).item()

        st.success(f"🧪 Parkinson’s Probability: **{prob:.2%}**")

        if prob > 0.5:
            st.warning("⚠️ Model indicates Parkinson’s characteristics.")
        else:
            st.info("✅ Model indicates healthy speech patterns.")

    except Exception as e:
        st.error(str(e))
else:
    st.info("Please upload **both** files to continue.")


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared


In [ ]:
!cloudflared --version

In [ ]:
!streamlit run app.py --server.port 8503 &>/content/logs.txt &
!cloudflared tunnel --url http://localhost:8503

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------
# Feature Importance from NN weights
# ---------------------------------
weights = model.net[0].weight.detach().cpu().numpy()

importance = np.mean(np.abs(weights), axis=0)

feature_importance = pd.Series(
    importance,
    index=feature_columns
).sort_values(ascending=False)

# Plot top 10 features
plt.figure(figsize=(8, 5))
feature_importance.head(10).plot(kind="barh")
plt.xlabel("Importance Score")
plt.title("Top 10 Most Influential Voice Features")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Print top features
print("Top Influential Features:")
print(feature_importance.head(10))


In [ ]:
import seaborn as sns

key_features = [
    "MDVP:Jitter(%)",
    "MDVP:Shimmer",
    "HNR"
]

for feature in key_features:
    plt.figure(figsize=(5, 4))
    sns.boxplot(x=y, y=X[feature])
    plt.xlabel("Class (0 = Healthy, 1 = Parkinson's)")
    plt.ylabel(feature)
    plt.title(f"{feature} by Class")
    plt.tight_layout()
    plt.show()


In [ ]:
for feature in key_features:
    plt.figure(figsize=(6, 4))
    # Create a temporary DataFrame for plotting
    plot_data = pd.DataFrame({
        'feature_value': X[feature],
        'class': y
    })
    sns.histplot(
        data=plot_data, # Pass the DataFrame to the 'data' argument
        x='feature_value', # Reference the column name from 'plot_data'
        hue='class',       # Reference the column name from 'plot_data'
        bins=30,
        kde=True,
        element="step"
    )
plt.figure(figsize=(8, 5))
feature_importance.head(10).plot(kind="barh")
plt.xlabel("Importance Score")
plt.title("Top 10 Most Influential Voice Features")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_binarized, preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Healthy", "Parkinson's"]
)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from google.colab import files
import joblib
import torch

# ------------------------------
# 1️⃣ Upload test dataset
# ------------------------------
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Loaded file: {file_name}")

# ------------------------------
# 2️⃣ Load dataset (no header)
# ------------------------------
df = pd.read_csv(file_name, header=None)
print("Dataset shape:", df.shape)

# ------------------------------
# 3️⃣ Split into features (X)
#     If your test file has NO label column,
#     change this to: X = df
# ------------------------------
X_raw_test_data = df.iloc[:, :-1]

# Make sure X_raw_test_data has the same columns as UNIFIED_FEATURES
# This assumes the test data has the same structure as train_data.txt
if X_raw_test_data.shape[1] < len(UNIFIED_FEATURES):
    raise ValueError(
        f"The test file '{file_name}' has {X_raw_test_data.shape[1]} features, "
        f"which is fewer than the {len(UNIFIED_FEATURES)} expected. "
        "Cannot align without specific column mapping."
    )
elif X_raw_test_data.shape[1] > len(UNIFIED_FEATURES):
    print(
        f"Warning: The test file '{file_name}' has {X_raw_test_data.shape[1]} features, "
        f"but only {len(UNIFIED_FEATURES)} were expected. "
        f"Taking the first {len(UNIFIED_FEATURES)} features for alignment."
    )
    X_raw_test_data = X_raw_test_data.iloc[:, :len(UNIFIED_FEATURES)]

X_raw_test_data.columns = UNIFIED_FEATURES

# Handle potential NaNs introduced by feature alignment
X_raw_test_data.fillna(X_raw_test_data.mean(numeric_only=True), inplace=True)

# ------------------------------
# 4️⃣ Load trained model and scaler
# ------------------------------
# Ensure model, scaler, and feature_columns are loaded or available
# The notebook state shows these variables are already defined and loaded in previous cells.
# model = ParkinsonNet(input_dim=len(feature_columns)) # Assuming ParkinsonNet class is defined
# model.load_state_dict(torch.load("parkinsons_model.pth", map_location="cpu"))
# model.eval()

# scaler = joblib.load("scaler.pkl")
# feature_columns = joblib.load("features.pkl")

# Scale the test data
X_scaled = scaler.transform(X_raw_test_data)

# ------------------------------
# 5️⃣ Predict probabilities using PyTorch model
# ------------------------------
model.eval() # Set model to evaluation mode
with torch.no_grad():
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    outputs = model(X_tensor).squeeze()
    # The model already has a Sigmoid layer, so outputs are probabilities
    parkinsons_prob = outputs.numpy() * 100

# ------------------------------
# 6️⃣ Print results
# ------------------------------
print("\n--- Prediction Results ---")
for i, p in enumerate(parkinsons_prob):
    print(f"Row {i}: Parkinson's probability = {p:.2f}%")


In [ ]:
import pandas as pd

# Get the number of entries (rows) from the combined DataFrame X
num_entries = len(X)

print(f"The combined dataset has {num_entries} entries.")

In [ ]:
from google.colab import files
import pandas as pd

# Ask user to upload file
uploaded = files.upload()

# Get uploaded filename
file_name = list(uploaded.keys())[0]

print(f"Loaded file: {file_name}")

# Load dataset (CSV / TXT)
df = pd.read_csv(file_name)

# Split into features and labels (same logic you already use)
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Dataset shape:", df.shape)

